# Vector stores and semantic search



For this activity, we will manually create two vector storing classes to allow for easy semantic search across documents.

We will use two datasets:
* animal-fun-facts-dataset (ekohrt - GitHub)
* Game Reviews Dataset (Kaggle)

We begin by importing the necessary modules, the majority aid dataset importation, while the cosine_similarity is strictly used to facilitate the search functions.

In [1]:
from sentence_transformers import SentenceTransformer

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import kagglehub
from kagglehub import KaggleDatasetAdapter

## Part I: Basic vector store implementation

These first implementations are the following:

* Document - Serves as a container for both a document's raw text, and its metadata.
* SearchResult - Holds the score of semantic searches along with its document.
* VectorStore - The actual vector store class, this holds a list of documents in which we will perform our searches, implements functions for adding and doing the semantic searches.

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        
        embeddings = self.embedding_model.encode([doc.text for doc in documents])
        self.embeddings.extend(embeddings)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode(query).reshape(1, -1)

        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [SearchResult(document=self.documents[i], score=similarities[i]) for i in top_indices]

## Activity

For the first part, we begin by importing our dataset from GitHub and transform it into a dataframe using pandas.

In [3]:
url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
df = pd.read_csv(url)

df

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark
...,...,...,...,...,...
7729,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"Like many reptiles, the incubation temperature...",NaN,/wiki/Spilotes_pullatus
7730,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"The yellow rat snake, or chicken snake, is kno...",NaN,/wiki/Spilotes_pullatus
7731,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,"Like pythons and boas, rat snakes are constric...",NaN,/wiki/Spilotes_pullatus
7732,yellow rat snake,https://seaworld.org/animals/facts/reptiles/ye...,Yellow rat snakes spend much time underground ...,NaN,/wiki/Spilotes_pullatus


After visualizing its columns, we manually create a document list to assign it later to our vector store.

We also make sure to skip any documents in which the text is empty, as there is no reason to include them in our list.

As metadata, we add all of the accompanying columns, such as the animal's name, the source of the fun fact, and links for media and wikipedia. 

In [4]:
AnimalFactsList = []

for _, AnimalFactRow in df.iterrows():
    if pd.notna(AnimalFactRow["text"]):
        AnimalFactsList.append(Document(
            text=AnimalFactRow["text"],
            metadata={
                "animal_name": AnimalFactRow["animal_name"],
                "source": AnimalFactRow["source"],
                "media_link": AnimalFactRow["media_link"],
                "wikipedia_link": AnimalFactRow["wikipedia_link"]
            }
        ))

We then proceed to import an embedding model. We use all-MiniLM-L6-v2 as it is commonly used.

We instantiate an object from our class and consequently add our document list.

This internal add function will also comput the document's embeddings and store them inside the class for seamless searching.

We finalize this section by transforming our internal embedding list into a number array, this is done so that it is compatible with the cosine_similarity function inside semantic search.

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

AnimalFactVectorStore = VectorStore(embedding_model=model)

AnimalFactVectorStore.add_documents(documents=AnimalFactsList)
AnimalFactVectorStore.embeddings = np.array(AnimalFactVectorStore.embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

After computing the data, we can define a function to cleanly print the results from searches.

In [6]:
def printResults(results):
    for i, r in enumerate(results):
        print(f"# {i + 1}. [{r.score:.2f}] {r.document.text} {r.document.metadata}\n")

And we can now do some queries and see the top 5 results from different custom documents.

In [7]:
results = AnimalFactVectorStore.search("they run very fast", top_k=5)
printResults(results)

# 1. [0.71] They can run at a speed of 15mph {'animal_name': 'polecat', 'source': 'https://a-z-animals.com/animals/polecat/', 'media_link': nan, 'wikipedia_link': '/wiki/Polecat'}

# 2. [0.68] They can run as fast as 45 mph. {'animal_name': 'jackrabbit', 'source': 'https://a-z-animals.com/animals/jackrabbit/', 'media_link': nan, 'wikipedia_link': '/wiki/Hare'}

# 3. [0.67] They can run as fast as 35 kph..
They are also extremely agile, which helps them run away from predators. {'animal_name': 'capybara', 'source': 'https://factanimal.com/capybara/', 'media_link': nan, 'wikipedia_link': '/wiki/Capybara'}

# 4. [0.64] They are very well coordinated and can run and jump by {'animal_name': 'puppy', 'source': 'https://www.animalfactsencyclopedia.com/Puppy-facts.html', 'media_link': nan, 'wikipedia_link': '/wiki/Puppy'}

# 5. [0.64] They can run up a tree trunk surprisingly fast. {'animal_name': 'gooty sapphire tarantula', 'source': 'https://a-z-animals.com/animals/gooty-sapphire-tarantula/'

In [8]:
results = AnimalFactVectorStore.search("are very good climbers", top_k=5)
printResults(results)

# 1. [0.60] Short claws make them good tree climbers! {'animal_name': 'north american black bear', 'source': 'https://a-z-animals.com/animals/north-american-black-bear/', 'media_link': nan, 'wikipedia_link': '/wiki/American_black_bear'}

# 2. [0.60] Black bears, including cinnamon bears, are excellent climbers, good runners, and powerful swimmers. {'animal_name': 'cinnamon bear', 'source': 'https://seaworld.org/animals/facts/mammals/cinnamon-bear/', 'media_link': nan, 'wikipedia_link': '/wiki/Cinnamon_bear'}

# 3. [0.56] Caracals are excellent climbers.
Though they spend most of their time on the ground, caracals are able to climb trees with ease, especially when hunting nesting birds. {'animal_name': 'caracal', 'source': 'https://factanimal.com/caracal/', 'media_link': nan, 'wikipedia_link': '/wiki/Caracal'}

# 4. [0.52] Corn snakes are partly arboreal and are excellent climbers. {'animal_name': 'corn snake', 'source': 'https://a-z-animals.com/animals/corn-snake/', 'media_link': nan, 

In [9]:
results = AnimalFactVectorStore.search("they inject their poison", top_k=5)
printResults(results)

# 1. [0.62] They inject hosts with a chemical that stops them from feeling the pain of the bite {'animal_name': 'tick', 'source': 'https://a-z-animals.com/animals/tick/', 'media_link': nan, 'wikipedia_link': '/wiki/Tick'}

# 2. [0.60] They are poisonous, not venomous..
This means that they do not inject their toxins into others, like snakes, they instead have to be consumed or licked. {'animal_name': 'poison dart frog', 'source': 'https://factanimal.com/poison-dart-frog/', 'media_link': nan, 'wikipedia_link': '/wiki/Poison_dart_frog'}

# 3. [0.59] They get toxins from their prey to use it against predators. {'animal_name': 'nudibranch', 'source': 'https://a-z-animals.com/animals/nudibranch/', 'media_link': nan, 'wikipedia_link': '/wiki/Nudibranch'}

# 4. [0.57] They secrete a milky poisonous liquid that can make many animals sick. {'animal_name': 'american toad', 'source': 'https://a-z-animals.com/animals/american-toad/', 'media_link': nan, 'wikipedia_link': '/wiki/American_toad'}

# 5

In [10]:
results = AnimalFactVectorStore.search("are famous for walking in one leg", top_k=5)
printResults(results)

# 1. [0.55] Sleeps on just one leg! {'animal_name': 'flamingo', 'source': 'https://a-z-animals.com/animals/flamingo/', 'media_link': nan, 'wikipedia_link': '/wiki/Flamingo'}

# 2. [0.54] It walked on two legs and leaned forward {'animal_name': 'suchomimus', 'source': 'https://a-z-animals.com/animals/suchomimus/', 'media_link': nan, 'wikipedia_link': '/wiki/Suchomimus'}

# 3. [0.50] Can have a leg span of nearly 2 meters! {'animal_name': 'king crab', 'source': 'https://a-z-animals.com/animals/king-crab/', 'media_link': nan, 'wikipedia_link': '/wiki/King_crab'}

# 4. [0.47] It is not entirely clear as to why flamingos stand on one leg.
You may have seen flamingos standing on one leg in the water. Researchers are not fully certain as to why they do so. {'animal_name': 'flamingo', 'source': 'https://factanimal.com/flamingo/', 'media_link': nan, 'wikipedia_link': '/wiki/Flamingo'}

# 5. [0.47] Long-legged giraffes walk with the limbs on one side of the body lifted at the same time. This gai

In [11]:
results = AnimalFactVectorStore.search("they sleep during the day", top_k=5)
printResults(results)

# 1. [0.94] They sleep during the day and are awake at night {'animal_name': 'syrian hamster', 'source': 'https://www.animalfactsencyclopedia.com/Syrian-hamster.html', 'media_link': nan, 'wikipedia_link': '/wiki/Golden_hamster'}

# 2. [0.69] These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of food, grooming, and resting. {'animal_name': 'coatimundi', 'source': 'https://seaworld.org/animals/facts/mammals/coatimundi/', 'media_link': nan, 'wikipedia_link': '/wiki/Coati'}

# 3. [0.65] but degus are not nocturnal, they are awake during the day {'animal_name': 'degu', 'source': 'https://www.animalfactsencyclopedia.com/Degu-facts.html', 'media_link': nan, 'wikipedia_link': '/wiki/Common_degu'}

# 4. [0.64] They are completely nocturnal .
They leave the roost to feed when it is dark and will wait for the moon to go down. {'animal_name': 'vampire bat', 'source': 'https://factanimal.com/vampire-bat/', 'media_link': nan, 'wikipe

We can observe that it worked perfectly, achieving great similarity results and simulating what a basic search engine does.

## Part II: Filtering by metadata

For the second part of this activity, we declare another vectore store class but with the filtering capability.

In the search function, we now pass an additional argument of metadata filters, which are used to filter the matching indexes, and then returned as the top 5 most similar filtered embeddings.

In [12]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        
        embeddings = self.embedding_model.encode([doc.text for doc in documents])
        self.embeddings.extend(embeddings)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode(query).reshape(1, -1)

        if metadata_filter:
            filtered_indices = [
                i for i, doc in enumerate(self.documents)
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
            filtered_embeddings = np.array([self.embeddings[i] for i in filtered_indices])
        else:
            filtered_indices = list(range(len(self.documents)))
            filtered_embeddings = self.embeddings

        similarities = cosine_similarity(query_embedding, filtered_embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]

        return [SearchResult(document=self.documents[filtered_indices[i]], score=similarities[i]) for i in top_indices]


## Activity

For the activity part, we import the Video Game Reviews dataset from Kaggle and print its resulting dataframe.

In [13]:
gr_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "vinayaks0n1/metacritic-game-reviews",
    "Metacriticgames8.csv"
)

print(gr_df.head())

                                              Review  Rating  Platform
0  Definitely one of the worst sequels I've ever ...       1  XBOX 360
1  The game is full of references, black humor an...       0  XBOX 360
2  Very nice shooter, in particular two players s...       9  XBOX 360
3  You know how people say that they play single ...       2  XBOX 360
4  Very bitter-sweet, similar to BL1. The world i...       6  XBOX 360


Similar to the past section, we create a list of documents using the other columns, in this case being rating and platform. We also use review as our main text.

In [14]:
GameReviewsList = []

for _, GameReviewRow in gr_df.iterrows():
    if pd.notna(GameReviewRow["Review"]):
        GameReviewsList.append(Document(
            text=GameReviewRow["Review"],
            metadata={
                "rating": GameReviewRow["Rating"],
                "platform": GameReviewRow["Platform"]
            }
        ))

We now create an instance of our filtered vectore sotore class and add the documents, followed by converting the embeddings into number format.

In [15]:
GameReviewFilteredVectorStore = FilteredVectorStore(embedding_model=model)

GameReviewFilteredVectorStore.add_documents(documents=GameReviewsList)
GameReviewFilteredVectorStore.embeddings = np.array(GameReviewFilteredVectorStore.embeddings)

We finally proceed to do some search queries, using filters.

In [16]:
results = GameReviewFilteredVectorStore.search("poor gameplay", top_k=5, metadata_filter={"platform": "PLAYSTATION 5"})
printResults(results)

# 1. [0.61] Bad graphics, worst optimization, bad storytelling, booooring. DON'T WASTE YOUR MONEY ON THIS TRASH I'm embarrassed to say i even played this horrible game {'rating': 0, 'platform': 'PLAYSTATION 5'}

# 2. [0.61] Poor gameplay, too much sexuality in game. Optimization is poor. Definitely not recommending. {'rating': 0, 'platform': 'PLAYSTATION 5'}

# 3. [0.59] Bad story and beat gameplay. {'rating': 0, 'platform': 'PLAYSTATION 5'}

# 4. [0.58] A crap, without gameplay, bad graphics and rotten gameplay, when playing, because it is a crap {'rating': 0, 'platform': 'PLAYSTATION 5'}

# 5. [0.56] History garbage but the gameplay and good {'rating': 3, 'platform': 'PLAYSTATION 5'}



In [17]:
results = GameReviewFilteredVectorStore.search("amazing story", top_k=5, metadata_filter={"rating": 9})
printResults(results)

# 1. [0.40] AWESOME GAME {'rating': 9, 'platform': 'PLAYSTATION 5'}

# 2. [0.39] Amazing game, good and satisfying gameplay but I feel like something is missing in the story, I can't really explain it {'rating': 9, 'platform': 'PLAYSTATION 5'}

# 3. [0.38] Simply amazing, simple and very fun, very much.Simple but intriguing story, I had never played but it's really amazing. {'rating': 9, 'platform': 'PC'}

# 4. [0.36] Excellent gaming experience. Everything was there: interesting story, perfect gameplay ... Bravo to developers for this work that honors the video game industry. {'rating': 9, 'platform': 'PLAYSTATION 5'}

# 5. [0.36] Everything the first one had to offer but better! Great story with one of the protagonist (the character I actually chose in part 1), being the leader now. {'rating': 9, 'platform': 'XBOX 360'}



In [18]:
results = GameReviewFilteredVectorStore.search("online multiplayer", top_k=5, metadata_filter={"platform": "XBOX 360"})
printResults(results)

# 1. [0.51] I have played this game offline till the end of the game. The story was great, the weapons feel amazing and there are a lot of different enemies. The best thing of the game is the voice acting and the interactions between friends and enemies. I was impressed and I was about to give this game a 8 or higher. But after I went online I lost my save.. and I was not happy with that. Thats something 2K needs to fix in the future! {'rating': 7, 'platform': 'XBOX 360'}

# 2. [0.47] To start with the good the single player campaign is amazingly well written and varied. The characters are great, and you really get connected to them as you play. Online is also often a lot of fun (when it works.) The bad: To buy all the properties in campaign you need something like 300 times the amount of money you'd accumulate during a normal game if you hadn't spent a dime on anything else. The worst: Online is completely broken. Rockstar is trying to make an IAP-based online game so they continually

In [19]:
results = GameReviewFilteredVectorStore.search("seamless world", top_k=5, metadata_filter={"platform": "PLAYSTATION 5"})
printResults(results)

# 1. [0.34] The best video game in the world {'rating': 10, 'platform': 'PLAYSTATION 5'}

# 2. [0.33] Short story: only linear part of FF7 and no open world This linearity makes FF7 feel restrictive and dated by today's standards. While the story itself is decent, it's disappointing not being able to explore the game's imaginative world more freely. Once you leave Midgar, the areas you travel through serve as little more than momentary backdrops along a predetermined track. The pacing also suffers when you're unable to take a break from the main storyline quests. {'rating': 3, 'platform': 'PLAYSTATION 5'}

# 3. [0.33] The game is scripted to death, it should have been a TV series, so at least you could browse a smartphone 50% of time during another "exciting" sequence of walking-talking along a corridor where you can move only in one direction... It's unfortunate as the visuals, the world and the ambience are pretty well done... {'rating': 5, 'platform': 'PLAYSTATION 5'}

# 4. [0.33] G

In [20]:
results = GameReviewFilteredVectorStore.search("soundtrack", top_k=5, metadata_filter={"rating": 2})
printResults(results)

# 1. [0.27] Bad game for those who actually play final fantasy games. Graphics, Square Enix put a lot, and I mean a lot, of time and resources in making the characters look beautiful but everything is complete sh**. They literally put a JPEG in 1 area. But it does have great cutscenes and made a 2D world in 3D showing off more then what was originally there in the PS1 version. Sound, Has almost the same music as the PS1 version the difference is the Remake has more instruments. While the music is not bad the surrounding sounds, voices, etc. seem to not know which needs to be louder. You could be done with an objective and listening to what the game wants you to do next but you won’t be able to hear it over the people in the background talking about their day and how they hate the Shinra company, this way too often. Gameplay, Gameplay is 3rd person hack n’slash with a menu you can use to pause the fight so you can choose what magic or item you want to use. If you’ve played Kingdom Heart

## Final reflections

In conclusion, this activity was a great way to learn how vector storing works. This not so challenging notebook allowed me to learn how search engines like google work. Before this class, I had no idea how computers easily found semantic similarities between texts.

What I also liked was that we additionally created a filter version, which shows how filters are applied to searches.

Having small 'toy' datasets gave us really good results, and it makes me wonder how efficient a search would be if I had imported a larger dataset with a bigger quantity of metadata. And this easy implementation would have allowed me to do it in no time, which is pleasing.